In [33]:
import pandas as pd

In [34]:
# read clean data
cleaned_df = pd.read_csv("data/cleaned_data.csv", keep_default_na=False)

# reconstruct mask mandate logic
mandate_df = pd.read_csv("data/mandate_start_dates.csv")

states_date = {
    state: [pd.to_datetime(date)] 
    for state, date in zip(mandate_df["RegionName"], mandate_df["Date"])
}

def mandates_convert(row):
    endtime = pd.to_datetime(row['endtime'], errors='coerce')
    state = row['state']

    if state not in states_date:
        return 0
    if pd.isna(endtime):
        return 0
        
    if states_date[state][0] <= endtime:
        return 1
    else:
        return 0
    
cleaned_df['within_mandate_period'] = cleaned_df.apply(mandates_convert, axis=1)

In [35]:
# one-hot encoding
convert_into_dummy_cols = [
    'state', 'gender', 'i9_health', 'employment_status', 'i11_health',
    'WCRex1', 'WCRex2', 'PHQ4_1', 'PHQ4_2', 'PHQ4_3', 'PHQ4_4',
    "d1_comorbidities"
]

# to prevent get_dummies from error, make sure that these columns are all str
for col in convert_into_dummy_cols:
    dummy = pd.get_dummies(cleaned_df[col], prefix=col, drop_first=True, dtype=int)
    cleaned_df = pd.concat([cleaned_df, dummy], axis=1)
    cleaned_df = cleaned_df.drop(col, axis=1)

In [36]:
# save preprocess data
cleaned_df.to_csv("data/preprocessing_data.csv", index=False)